In [1]:
import sys
import os
import pyspark

print("Python :", sys.version)
print("PySpark :", pyspark.__version__)
print("Répertoire courant :", os.getcwd())

print("PGHOST :", os.getenv("PGHOST"))
print("PGPORT :", os.getenv("PGPORT"))
print("PGDATABASE :", os.getenv("PGDATABASE"))
print("PGUSER :", os.getenv("PGUSER"))

Python : 3.8.10 (default, Sep 11 2024, 16:02:53) 
[GCC 9.4.0]
PySpark : 3.5.3
Répertoire courant : /opt/spark/work-dir/notebooks
PGHOST : postgres
PGPORT : 5432
PGDATABASE : mastodon
PGUSER : spark


In [2]:
import psycopg2
import os

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

tables = cursor.fetchall()

print("Connexion PostgreSQL réussie ✅")
print("\nTables disponibles :")

for table in tables:
    print("-", table[0])

cursor.close()
conn.close()

Connexion PostgreSQL réussie ✅

Tables disponibles :
- toots


In [3]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = 'toots'
    ORDER BY ordinal_position;
""")

columns = cursor.fetchall()

print("Structure de la table toots :\n")

for column_name, data_type in columns:
    print(f"{column_name:20} -> {data_type}")

cursor.close()
conn.close()

Structure de la table toots :

id                   -> bigint
toot_id              -> character varying
created_at           -> timestamp without time zone
user_id              -> character varying
username             -> character varying
content              -> text
language             -> character varying
hashtags             -> ARRAY
favourites_count     -> integer
reblogs_count        -> integer


In [5]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE language = 'en') AS english_posts,
        MIN(created_at) AS first_post,
        MAX(created_at) AS last_post
    FROM toots;
""")

result = cursor.fetchone()

print("Nombre total de toots :", result[0])
print("Toots en anglais      :", result[1])
print("Premier toot          :", result[2])
print("Dernier toot          :", result[3])

cursor.close()
conn.close()

Nombre total de toots : 61
Toots en anglais      : 55
Premier toot          : 2026-10-02 21:29:50.734192
Dernier toot          : 2026-10-08 13:24:45


In [6]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("03-mastodon-sentiment")
    .getOrCreate()
)

print("Version Spark :", spark.version)
print("Application   :", spark.sparkContext.appName)
print("Spark UI      :", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 13:25:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/08 13:25:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Version Spark : 3.5.3
Application   : 03-mastodon-sentiment
Spark UI      : http://9d066d2ef4fc:4041


In [7]:
import os

DATA_PATH = "../data/training.1600000.processed.noemoticon.csv"

print("Chemin :", os.path.abspath(DATA_PATH))
print("Fichier présent :", os.path.exists(DATA_PATH))

if os.path.exists(DATA_PATH):
    size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
    print(f"Taille : {size_mb:.2f} Mo")

Chemin : /opt/spark/work-dir/data/training.1600000.processed.noemoticon.csv
Fichier présent : True
Taille : 227.74 Mo


In [8]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    LongType,
    StringType
)

sentiment_schema = StructType([
    StructField("target", IntegerType(), True),
    StructField("tweet_id", LongType(), True),
    StructField("date", StringType(), True),
    StructField("flag", StringType(), True),
    StructField("user", StringType(), True),
    StructField("text", StringType(), True)
])

In [9]:
sentiment_raw = (
    spark.read
    .option("header", "false")
    .option("encoding", "ISO-8859-1")
    .schema(sentiment_schema)
    .csv(DATA_PATH)
)

In [10]:
sentiment_raw.printSchema()

sentiment_raw.show(5, truncate=False)

print("Nombre de lignes :", sentiment_raw.count())

root
 |-- target: integer (nullable = true)
 |-- tweet_id: long (nullable = true)
 |-- date: string (nullable = true)
 |-- flag: string (nullable = true)
 |-- user: string (nullable = true)
 |-- text: string (nullable = true)



+------+----------+----------------------------+--------+---------------+-------------------------------------------------------------------------------------------------------------------+
|target|tweet_id  |date                        |flag    |user           |text                                                                                                               |
+------+----------+----------------------------+--------+---------------+-------------------------------------------------------------------------------------------------------------------+
|0     |1467810369|Mon Apr 06 22:19:45 PDT 2009|NO_QUERY|_TheSpecialOne_|@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D|
|0     |1467810672|Mon Apr 06 22:19:49 PDT 2009|NO_QUERY|scotthamilton  |is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!    |
|0     |1467810917|Mon Apr 06 22:19:53 PDT 2009|NO

Nombre de lignes : 1600000


In [11]:
sentiment_raw.groupBy("target").count().orderBy("target").show()

+------+------+
|target| count|
+------+------+
|     0|800000|
|     4|800000|
+------+------+



In [12]:
from pyspark.sql import functions as F

sentiment = (
    sentiment_raw
    .select("target", "text")
    .withColumn(
        "label",
        F.when(F.col("target") == 4, F.lit(1.0))
         .otherwise(F.lit(0.0))
    )
)

In [13]:
sentiment.groupBy("label").count().orderBy("label").show()

+-----+------+
|label| count|
+-----+------+
|  0.0|800000|
|  1.0|800000|
+-----+------+



In [14]:
sentiment.show(5, truncate=False)

+------+-------------------------------------------------------------------------------------------------------------------+-----+
|target|text                                                                                                               |label|
+------+-------------------------------------------------------------------------------------------------------------------+-----+
|0     |@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D|0.0  |
|0     |is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!    |0.0  |
|0     |@Kenichan I dived many times for the ball. Managed to save 50%  The rest go out of bounds                          |0.0  |
|0     |my whole body feels itchy and like its on fire                                                                     |0.0  |
|0     |@nationwideclass no, it's not behaving at all. i'm mad. why am i here? beca

In [15]:
sentiment_clean = (
    sentiment

    # Passage en minuscules
    .withColumn(
        "clean_text",
        F.lower(F.col("text"))
    )

    # Conservation des négations
    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"\bcan't\b", "can not")
    )
    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"\bwon't\b", "will not")
    )
    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"n't\b", " not")
    )

    # Suppression des URL
    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"http\S+|www\S+", " ")
    )

    # Suppression des mentions @utilisateur
    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"@\w+", " ")
    )

    # Suppression de la ponctuation et des chiffres
    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"[^a-z\s]", " ")
    )

    # Normalisation des espaces
    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"\s+", " ")
    )

    # Suppression des espaces début/fin
    .withColumn(
        "clean_text",
        F.trim(F.col("clean_text"))
    )
)

In [16]:
sentiment_clean.select(
    "text",
    "clean_text",
    "label"
).show(10, truncate=False)

+-------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------+-----+
|text                                                                                                               |clean_text                                                                                                 |label|
+-------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------+-----+
|@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D|a that s a bummer you shoulda got david carr of third day to do it d                                       |0.0  |
|is upset that he can't update his Facebook by texting it... and might c

In [17]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

tokenizer = RegexTokenizer(
    inputCol="clean_text",
    outputCol="words",
    pattern=r"\s+",
    gaps=True,
    minTokenLength=2
)

In [18]:
tokenized = tokenizer.transform(sentiment_clean)

In [19]:
english_stopwords = StopWordsRemover.loadDefaultStopWords("english")

important_negations = {"no", "not", "nor"}

custom_stopwords = [
    word
    for word in english_stopwords
    if word not in important_negations
]

stopwords_remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words",
    stopWords=custom_stopwords
)

prepared = stopwords_remover.transform(tokenized)

In [20]:
prepared.select(
    "words",
    "filtered_words",
    "label"
).show(10, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+-----+
|words                                                                                                                          |filtered_words                                                                        |label|
+-------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+-----+
|[that, bummer, you, shoulda, got, david, carr, of, third, day, to, do, it]                                                     |[bummer, shoulda, got, david, carr, third, day]                                       |0.0  |
|[is, upset, that, he, can, not, update, his, facebook, by, texting, it, and, might, cry, as, result, school

In [24]:
train, test = sentiment_clean.randomSplit(
    [0.8, 0.2],
    seed=42
)

train_count = train.count()
test_count = test.count()

print("Train :", train_count)
print("Test  :", test_count)
print("Total :", train_count + test_count)

Train : 1280209
Test  : 319791
Total : 1600000


In [25]:
print("TRAIN")
train.groupBy("label").count().orderBy("label").show()

print("TEST")
test.groupBy("label").count().orderBy("label").show()

TRAIN


+-----+------+
|label| count|
+-----+------+
|  0.0|639767|
|  1.0|640442|
+-----+------+

TEST


+-----+------+
|label| count|
+-----+------+
|  0.0|160233|
|  1.0|159558|
+-----+------+



In [26]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression
import time

count_vectorizer = CountVectorizer(
    inputCol="filtered_words",
    outputCol="tf",
    vocabSize=30000,
    minDF=5
)

idf = IDF(
    inputCol="tf",
    outputCol="features"
)

logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

pipeline_lr = Pipeline(
    stages=[
        tokenizer,
        stopwords_remover,
        count_vectorizer,
        idf,
        logistic_regression
    ]
)

print("Pipeline Logistic Regression prêt ✅")

Pipeline Logistic Regression prêt ✅


In [27]:
start = time.time()

model_lr = pipeline_lr.fit(train)

training_time_lr = time.time() - start

print(f"Temps d'entraînement : {training_time_lr:.2f} secondes")

26/10/08 13:52:37 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Temps d'entraînement : 203.93 secondes


In [28]:
predictions_lr = model_lr.transform(test)

predictions_lr.select(
    "text",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

26/10/08 14:00:24 WARN DAGScheduler: Broadcasting large task binary with size 1081.1 KiB


+----------------------------------------------------------------------------------------------------------------------------------------+-----+----------+-----------------------------------------+
|text                                                                                                                                    |label|prediction|probability                              |
+----------------------------------------------------------------------------------------------------------------------------------------+-----+----------+-----------------------------------------+
|        my head feels like a bowling ball                                                                                               |0.0  |1.0       |[0.40974986363901394,0.590250136360986]  |
|     #canucks                                                                                                                           |0.0  |0.0       |[0.50131236718373,0.49868763281627004]   |
|     I do

In [29]:
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_lr = accuracy_evaluator.evaluate(predictions_lr)
f1_lr = f1_evaluator.evaluate(predictions_lr)
auc_lr = auc_evaluator.evaluate(predictions_lr)

print(f"Accuracy : {accuracy_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC ROC  : {auc_lr:.4f}")

26/10/08 14:01:24 WARN DAGScheduler: Broadcasting large task binary with size 1089.2 KiB
26/10/08 14:02:04 WARN DAGScheduler: Broadcasting large task binary with size 1089.2 KiB
26/10/08 14:02:31 WARN DAGScheduler: Broadcasting large task binary with size 1077.2 KiB


Accuracy : 0.7887
F1-score : 0.7886
AUC ROC  : 0.8606


In [30]:
confusion_lr = (
    predictions_lr
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

confusion_lr.show()

26/10/08 14:03:05 WARN DAGScheduler: Broadcasting large task binary with size 1087.9 KiB


+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  0.0|       0.0|123301|
|  0.0|       1.0| 36932|
|  1.0|       0.0| 30638|
|  1.0|       1.0|128920|
+-----+----------+------+



26/10/08 14:03:46 WARN DAGScheduler: Broadcasting large task binary with size 1065.0 KiB


In [31]:
from pyspark.ml.classification import NaiveBayes

naive_bayes = NaiveBayes(
    featuresCol="features",
    labelCol="label",
    modelType="multinomial",
    smoothing=1.0
)

pipeline_nb = Pipeline(
    stages=[
        tokenizer,
        stopwords_remover,
        count_vectorizer,
        idf,
        naive_bayes
    ]
)

print("Pipeline Naive Bayes prêt ✅")

Pipeline Naive Bayes prêt ✅


In [32]:
start = time.time()

model_nb = pipeline_nb.fit(train)

training_time_nb = time.time() - start

print(f"Temps d'entraînement Naive Bayes : {training_time_nb:.2f} secondes")

Temps d'entraînement Naive Bayes : 137.40 secondes


In [33]:
predictions_nb = model_nb.transform(test)

predictions_nb.select(
    "text",
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

26/10/08 14:10:14 WARN DAGScheduler: Broadcasting large task binary with size 1309.5 KiB


+----------------------------------------------------------------------------------------------------------------------------------------+-----+----------+------------------------------------------+
|text                                                                                                                                    |label|prediction|probability                               |
+----------------------------------------------------------------------------------------------------------------------------------------+-----+----------+------------------------------------------+
|        my head feels like a bowling ball                                                                                               |0.0  |0.0       |[0.9826689691068882,0.01733103089311195]  |
|     #canucks                                                                                                                           |0.0  |0.0       |[0.9581902759821437,0.041809724017856285] |
|    

In [34]:
accuracy_nb = accuracy_evaluator.evaluate(predictions_nb)
f1_nb = f1_evaluator.evaluate(predictions_nb)
auc_nb = auc_evaluator.evaluate(predictions_nb)

print(f"Accuracy : {accuracy_nb:.4f}")
print(f"F1-score : {f1_nb:.4f}")
print(f"AUC ROC  : {auc_nb:.4f}")

26/10/08 14:11:01 WARN DAGScheduler: Broadcasting large task binary with size 1317.6 KiB
26/10/08 14:11:39 WARN DAGScheduler: Broadcasting large task binary with size 1317.6 KiB
26/10/08 14:12:13 WARN DAGScheduler: Broadcasting large task binary with size 1305.5 KiB


Accuracy : 0.7630
F1-score : 0.7630
AUC ROC  : 0.5331


In [35]:
confusion_nb = (
    predictions_nb
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
)

confusion_nb.show()

26/10/08 14:12:50 WARN DAGScheduler: Broadcasting large task binary with size 1316.3 KiB


+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  0.0|       0.0|122434|
|  0.0|       1.0| 37799|
|  1.0|       0.0| 37992|
|  1.0|       1.0|121566|
+-----+----------+------+



26/10/08 14:13:22 WARN DAGScheduler: Broadcasting large task binary with size 1293.4 KiB


In [36]:
auc_probability_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="probability",
    metricName="areaUnderROC"
)

auc_lr_probability = auc_probability_evaluator.evaluate(predictions_lr)
auc_nb_probability = auc_probability_evaluator.evaluate(predictions_nb)

print(f"AUC Logistic Regression : {auc_lr_probability:.4f}")
print(f"AUC Naive Bayes          : {auc_nb_probability:.4f}")

26/10/08 14:15:27 WARN DAGScheduler: Broadcasting large task binary with size 1081.5 KiB
26/10/08 14:16:11 WARN DAGScheduler: Broadcasting large task binary with size 1309.8 KiB


AUC Logistic Regression : 0.8606
AUC Naive Bayes          : 0.8313


In [37]:
import os

MODEL_PATH = "../models/sentiment_lr"

os.makedirs("../models", exist_ok=True)

model_lr.write().overwrite().save(MODEL_PATH)

print("Modèle sauvegardé ✅")
print("Chemin :", os.path.abspath(MODEL_PATH))

Modèle sauvegardé ✅
Chemin : /opt/spark/work-dir/models/sentiment_lr


In [38]:
import psycopg2
import os

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        toot_id,
        created_at,
        username,
        language,
        content
    FROM toots
    WHERE language = 'en'
      AND content IS NOT NULL
    ORDER BY created_at;
""")

rows = cursor.fetchall()

cursor.close()
conn.close()

print("Toots anglais récupérés :", len(rows))

Toots anglais récupérés : 81


In [39]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

toots_schema = StructType([
    StructField("toot_id", StringType(), False),
    StructField("created_at", TimestampType(), True),
    StructField("username", StringType(), True),
    StructField("language", StringType(), True),
    StructField("content", StringType(), True)
])

toots_df = spark.createDataFrame(rows, schema=toots_schema)

toots_df.show(5, truncate=False)

+-------+--------------------------+--------+--------+---------------------------------------+
|toot_id|created_at                |username|language|content                                |
+-------+--------------------------+--------+--------+---------------------------------------+
|1005   |2026-10-02 21:29:50.734192|bob     |en      |Data engineering project with Mastodon |
|1003   |2026-10-03 21:29:50.734192|alice   |en      |AI and machine learning are interesting|
|1004   |2026-10-03 21:29:50.734192|charlie |en      |PostgreSQL works great with Spark      |
|1002   |2026-10-04 20:29:50.734192|bob     |en      |Learning Kafka with Spark streaming    |
|1001   |2026-10-04 21:29:50.734192|alice   |en      |I love Apache Spark and Data Science   |
+-------+--------------------------+--------+--------+---------------------------------------+
only showing top 5 rows



In [40]:
toots_clean = (
    toots_df

    .withColumn(
        "clean_text",
        F.lower(F.col("content"))
    )

    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"\bcan't\b", "can not")
    )
    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"\bwon't\b", "will not")
    )
    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"n't\b", " not")
    )

    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"http\S+|www\S+", " ")
    )

    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"@\w+", " ")
    )

    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"[^a-z\s]", " ")
    )

    .withColumn(
        "clean_text",
        F.regexp_replace("clean_text", r"\s+", " ")
    )

    .withColumn(
        "clean_text",
        F.trim(F.col("clean_text"))
    )
)

In [41]:
mastodon_predictions = model_lr.transform(toots_clean)

In [42]:
mastodon_predictions.select(
    "toot_id",
    "username",
    "content",
    "prediction",
    "probability"
).show(20, truncate=False)

26/10/08 14:22:39 WARN DAGScheduler: Broadcasting large task binary with size 1066.6 KiB
26/10/08 14:22:39 WARN DAGScheduler: Broadcasting large task binary with size 1066.6 KiB


+------------------+------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+------------------------------------------+
|toot_id           |username                            |content                                                                                                                                                                                        

In [43]:
from pyspark.ml.functions import vector_to_array

mastodon_results = (
    mastodon_predictions
    .withColumn(
        "p_positive",
        vector_to_array("probability")[1]
    )
    .withColumn(
        "predicted_sentiment",
        F.when(F.col("prediction") == 1.0, "positive")
         .otherwise("negative")
    )
)

In [44]:
mastodon_results.select(
    "toot_id",
    "username",
    "content",
    "predicted_sentiment",
    F.round("p_positive", 3).alias("p_positive")
).show(20, truncate=False)

26/10/08 14:29:18 WARN DAGScheduler: Broadcasting large task binary with size 1070.2 KiB
26/10/08 14:29:18 WARN DAGScheduler: Broadcasting large task binary with size 1070.2 KiB


+------------------+------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+----------+
|toot_id           |username                            |content                                                                                                                                                                                                               

In [45]:
mastodon_results = (
    mastodon_results
    .withColumn(
        "sentiment_display",
        F.when(F.col("p_positive") >= 0.65, "positive")
         .when(F.col("p_positive") <= 0.35, "negative")
         .otherwise("uncertain")
    )
)

In [46]:
mastodon_results.groupBy(
    "predicted_sentiment"
).count().orderBy("predicted_sentiment").show()

26/10/08 14:32:45 WARN DAGScheduler: Broadcasting large task binary with size 1078.8 KiB


+-------------------+-----+
|predicted_sentiment|count|
+-------------------+-----+
|           negative|   14|
|           positive|   67|
+-------------------+-----+



26/10/08 14:32:46 WARN DAGScheduler: Broadcasting large task binary with size 1060.1 KiB


In [47]:
mastodon_results.groupBy(
    "sentiment_display"
).count().orderBy("sentiment_display").show()

26/10/08 14:32:55 WARN DAGScheduler: Broadcasting large task binary with size 1084.2 KiB


+-----------------+-----+
|sentiment_display|count|
+-----------------+-----+
|         negative|    8|
|         positive|   63|
|        uncertain|   10|
+-----------------+-----+



26/10/08 14:32:56 WARN DAGScheduler: Broadcasting large task binary with size 1062.6 KiB


In [48]:
mastodon_results.select(
    "content",
    "predicted_sentiment",
    F.round("p_positive", 3).alias("p_positive"),
    "sentiment_display"
).filter(
    F.col("sentiment_display") == "uncertain"
).show(10, truncate=False)

26/10/08 14:33:05 WARN DAGScheduler: Broadcasting large task binary with size 1115.9 KiB
26/10/08 14:33:05 WARN DAGScheduler: Broadcasting large task binary with size 1115.9 KiB
26/10/08 14:33:06 WARN DAGScheduler: Broadcasting large task binary with size 1115.9 KiB


+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------+----------+-----------------+
|content                                                                                                                                                                                                                                                                                                                                                                                                                             

In [49]:
demo_toot_ids = ["1001", "1002", "1003", "1004", "1005"]

mastodon_real_results = (
    mastodon_results
    .filter(~F.col("toot_id").isin(demo_toot_ids))
)

In [50]:
print("Résultats totaux :", mastodon_results.count())
print("Vrais toots Mastodon :", mastodon_real_results.count())

Résultats totaux : 81
Vrais toots Mastodon : 76


In [51]:
mastodon_real_results.groupBy(
    "sentiment_display"
).count().orderBy("sentiment_display").show()

26/10/08 14:35:43 WARN DAGScheduler: Broadcasting large task binary with size 1087.0 KiB


+-----------------+-----+
|sentiment_display|count|
+-----------------+-----+
|         negative|    8|
|         positive|   60|
|        uncertain|    8|
+-----------------+-----+



26/10/08 14:35:43 WARN DAGScheduler: Broadcasting large task binary with size 1063.9 KiB


In [52]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS toot_sentiments (
        toot_id VARCHAR(100) PRIMARY KEY,
        predicted_label SMALLINT NOT NULL,
        predicted_sentiment VARCHAR(20) NOT NULL,
        sentiment_display VARCHAR(20) NOT NULL,
        p_positive DOUBLE PRECISION NOT NULL,
        model_name VARCHAR(100) NOT NULL,
        predicted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

        CONSTRAINT fk_toot_sentiment
            FOREIGN KEY (toot_id)
            REFERENCES toots(toot_id)
            ON DELETE CASCADE
    );
""")

conn.commit()

cursor.close()
conn.close()

print("Table toot_sentiments créée ✅")

Table toot_sentiments créée ✅


In [53]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

for table in cursor.fetchall():
    print(table[0])

cursor.close()
conn.close()

streaming_posts_per_hour
streaming_user_stats_hourly
toot_sentiments
toots


In [54]:
results_to_insert = (
    mastodon_real_results
    .select(
        "toot_id",
        F.col("prediction").cast("int").alias("predicted_label"),
        "predicted_sentiment",
        "sentiment_display",
        "p_positive"
    )
    .collect()
)

rows_to_insert = [
    (
        row["toot_id"],
        row["predicted_label"],
        row["predicted_sentiment"],
        row["sentiment_display"],
        float(row["p_positive"]),
        "logistic_regression_tfidf_sentiment140_v1"
    )
    for row in results_to_insert
]

print("Résultats prêts à être enregistrés :", len(rows_to_insert))

26/10/08 14:39:18 WARN DAGScheduler: Broadcasting large task binary with size 1072.7 KiB


Résultats prêts à être enregistrés : 76


In [55]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.executemany("""
    INSERT INTO toot_sentiments (
        toot_id,
        predicted_label,
        predicted_sentiment,
        sentiment_display,
        p_positive,
        model_name
    )
    VALUES (%s, %s, %s, %s, %s, %s)

    ON CONFLICT (toot_id)
    DO UPDATE SET
        predicted_label = EXCLUDED.predicted_label,
        predicted_sentiment = EXCLUDED.predicted_sentiment,
        sentiment_display = EXCLUDED.sentiment_display,
        p_positive = EXCLUDED.p_positive,
        model_name = EXCLUDED.model_name,
        predicted_at = CURRENT_TIMESTAMP;
""", rows_to_insert)

conn.commit()

cursor.close()
conn.close()

print("Prédictions enregistrées dans PostgreSQL ✅")

Prédictions enregistrées dans PostgreSQL ✅


In [56]:
conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    database=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD")
)

cursor = conn.cursor()

cursor.execute("""
    SELECT
        sentiment_display,
        COUNT(*)
    FROM toot_sentiments
    GROUP BY sentiment_display
    ORDER BY sentiment_display;
""")

print("Résultats enregistrés :")

for sentiment_name, count in cursor.fetchall():
    print(f"{sentiment_name:10} : {count}")

cursor.close()
conn.close()

Résultats enregistrés :
negative   : 8
positive   : 60
uncertain  : 8
